# Start Here | Water Quality Research

สำหรับผู้รับช่วง: รันจากข้อมูลภาคสนามที่คลีนแล้วและไฟล์ดาวเทียมที่ทีม export ไว้ ไม่ต้องมีสิทธิ์ Earth Engine

This first check assembles the model dataset, validates predictions from the six locked model configurations, and tests the archived spatial inputs and selected pixel comparisons. It does not render every manuscript figure. Use a fresh CPU runtime.


## Load the team data

Open the authorized Drive folder and add the shared ZIP to **My Drive** as a shortcut, or upload your copy there. Set `ARCHIVE_PATH` if the file is in a subfolder. This cell verifies and unpacks the exact team snapshot to temporary Colab storage. No Earth Engine login is used. Save generated results to Drive before disconnecting.


In [ ]:
import hashlib
import os
import tempfile
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

MY_DRIVE = Path("/content/drive/MyDrive")
FULL_ARCHIVES = {
    "Water_Quality_Processed_Handoff_Windows_v2.zip": "bff939d07db4ca710e62b3af1ba55386d498768cfc82bda3afe58c7e59ce0adc",
    "Water_Quality_Processed_Handoff_Windows.zip": "324437bb6d0bb84c6bc1865a8ea1a172171974c6f039355deacc6a15ad7acc70",
    "Water_Quality_Processed_Handoff.zip": "1f1039b7a6d113685c2c95f840f2a475be5e936e02090bcf5283f2ba9b506773",
}
full_archive = next(
    (MY_DRIVE / name for name in FULL_ARCHIVES if (MY_DRIVE / name).is_file()),
    MY_DRIVE / "Water_Quality_Processed_Handoff_Windows_v2.zip",
)

ARCHIVE_PATH = full_archive
EXPECTED_SHA256 = FULL_ARCHIVES.get(ARCHIVE_PATH.name, "1f1039b7a6d113685c2c95f840f2a475be5e936e02090bcf5283f2ba9b506773")


HANDOFF_ROOT = Path("/content/wqr-handoff-" + EXPECTED_SHA256[:12])
marker = HANDOFF_ROOT / ".verified-archive-sha256"

if not (marker.is_file() and marker.read_text().strip() == EXPECTED_SHA256):
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            f"{ARCHIVE_PATH} was not found. Ask the project owner for the selected ZIP "
            "and point ARCHIVE_PATH to its location on the mounted Drive."
        )
    with tempfile.TemporaryDirectory(dir=HANDOFF_ROOT.parent) as tmp:
        tmp = Path(tmp)
        local_zip = tmp / ARCHIVE_PATH.name
        digest = hashlib.sha256()
        with ARCHIVE_PATH.open("rb") as src, local_zip.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
                dst.write(chunk)
        if digest.hexdigest() != EXPECTED_SHA256:
            raise ValueError("This ZIP differs from the verified snapshot. Check the filename and download.")
        extracted = tmp / "snapshot"
        with zipfile.ZipFile(local_zip) as archive:
            archive.extractall(extracted)
        (extracted / marker.name).write_text(EXPECTED_SHA256 + "\n")
        os.replace(extracted, HANDOFF_ROOT)

print("Team data ready:", HANDOFF_ROOT)


## Install and verify

Python 3.11 is the reference environment. The runner executes in a separate environment so package changes do not affect the notebook kernel. For a different runtime, the same pinned dependencies must be available; otherwise use the Python 3.11 local route in the team runbook.


In [ ]:
import subprocess
import sys

OUTPUT_ROOT = HANDOFF_ROOT / "run_output" / "first_check"
RUN_GLOBAL_SHAP = False

print("Runtime Python:", sys.version.split()[0])
env_dir = HANDOFF_ROOT / ".verification-env"
python = env_dir / "bin" / "python"
if not python.exists():
    subprocess.run([sys.executable, "-m", "venv", str(env_dir)], check=True)
subprocess.run([str(python), "-m", "pip", "install", "-q",
                "-r", str(HANDOFF_ROOT / "requirements.txt")], check=True)
command = [str(python), "-B", str(HANDOFF_ROOT / "tools/reproduce.py"),
           "--output", str(OUTPUT_ROOT)]
if RUN_GLOBAL_SHAP:
    command.append("--global-shap")
subprocess.run(command, cwd=HANDOFF_ROOT, check=True)


In [ ]:
from IPython.display import JSON, display

receipt = OUTPUT_ROOT / "verification.json"
display(JSON(data=receipt.read_text(encoding="utf-8")))


In [ ]:
# Optional: choose your own private destination and uncomment to persist results.
# import shutil
# destination = Path("/content/drive/MyDrive/wqr_my_results")
# shutil.copytree(OUTPUT_ROOT, destination, dirs_exist_ok=True)
print("Results for this run:", OUTPUT_ROOT)
